# 04 - Variantes Avanzadas: Bisecting K-Means, K-Medoids (PAM) y Datos Mixtos

## Curso de Machine Learning: Modelos No Supervisados

En los tres primeros cuadernos estudiamos la formulacion clasica del algoritmo de Lloyd (`KMeans`), las metricas de validacion interna para elegir $K^*$ y las tecnicas de perfilamiento de centroides. Sin embargo, al enfrentarnos a datos tabulares complejos en produccion surgen tres limitaciones estructurales del algoritmo estandar:

1. **Inconsistencia Jerarquica y Minimos Locales**: En K-Means clasico, los clusters obtenidos con $K=4$ no son necesariamente una subdivision limpia de los obtenidos con $K=3$, y cuando $K$ es elevado el algoritmo suele estancarse en minimos locales suboptimos.
2. **Fragilidad Extrema ante Outliers (Punto de Ruptura $0\%$)**: Dado que el centroide $\mu_k$ es una media aritmetica que minimiza distancias al cuadrado ($L_2^2$), una sola observacion corrupta o extrema desplaza el centroide hacia el vacio, haciendo que deje de representar a los miembros reales del grupo.
3. **Incompatibilidad con Variables Categoricas (Datos Mixtos)**: En bases de datos empresariales (banca, retail, salud), las tablas combinan variables continuas (edad, ingresos) con variables categoricas nominales (profesion, canal de pago, region). Aplicar K-Means sobre variables codificadas con *One-Hot Encoding* destruye la geometria del espacio.

En este cuaderno abordaremos las soluciones algoritmicas formales a estos tres problemas: **`BisectingKMeans`**, **`K-Medoids` (PAM)** y el agrupamiento sobre **Datos Mixtos (Distancia de Gower y K-Prototypes)**.

---

### Contenidos
1. Agrupamiento Divisivo Hibrido: `BisectingKMeans` vs. K-Means de Lloyd
2. Estrategias de Biseccion: Mayor Inercia (`biggest_inertia`) vs. Mayor Tamano (`largest_cluster`)
3. Robustez Estadistica: Del Centroide Sintetico ($\mu_k$) al Medoide Real ($m_k$) con K-Medoids (PAM)
4. Demostracion Empirica ante Outliers Severos: K-Means ($L_2^2$) vs. K-Medoids ($L_1$ / Manhattan)
5. El Problema de los Datos Mixtos: Por que Fracasa el One-Hot Encoding en K-Means
6. Solucion 1 para Datos Mixtos: Matriz de Disimilaridad de Gower + K-Medoids
7. Solucion 2 para Datos Mixtos: El Algoritmo **K-Prototypes** (Centroides + Modas con Peso $\gamma$)
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, BisectingKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, pairwise_distances

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Agrupamiento Divisivo Hibrido: `BisectingKMeans`

El algoritmo **Bisecting K-Means** (Steinbach et al., 2000), implementado nativamente en `sklearn.cluster.BisectingKMeans`, combina la velocidad de K-Means con la estructura ordenada del **Clustering Jerarquico Divisivo (Top-Down)**:

### Algoritmo Paso a Paso:
1. **Inicio ($k = 1$)**: Comienza con un unico gran cluster que contiene el $100\%$ de las observaciones $X$.
2. **Seleccion del Cluster a Dividir**: Elige uno de los clusters actuales para ser partido en dos, siguiendo un criterio (`bisecting_strategy`):
   - `'biggest_inertia'` (por defecto): Selecciona el cluster con mayor SSE / Inercia interna.
   - `'largest_cluster'`: Selecciona el cluster con mayor numero de observaciones.
3. **Biseccion Local ($K=2$)**: Aplica un K-Means binario ($K=2$) exclusivamente sobre los puntos de ese cluster seleccionado (repitiendo `n_init` veces para asegurar el mejor corte binario posible).
4. **Iteracion**: Repite los pasos 2 y 3 hasta alcanzar el numero deseado de clusters $K$.

### Ventajas Frente a K-Means Estandar:
- **Propiedad Jerarquica**: Cada paso refina la particion anterior sin reorganizar caoticamente los clusters que ya estaban bien compactos.
- **Menor Sensibilidad a la Inicializacion**: Resolver multiples problemas simples de $K=2$ es mucho menos propenso a caer en malos optimos locales que intentar posicionar simultaneamente $K=15$ centroides en todo el espacio.


In [ ]:
# Comparativa entre KMeans de Lloyd y BisectingKMeans a medida que aumenta K
X_blobs, _ = make_blobs(n_samples=1500, n_features=10, centers=12, cluster_std=1.4, random_state=42)
X_blobs_scaled = StandardScaler().fit_transform(X_blobs)

rango_k = range(2, 18)
inercia_lloyd = []
inercia_bisect_inertia = []
inercia_bisect_largest = []

for k in rango_k:
    km = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42).fit(X_blobs_scaled)
    bkm_in = BisectingKMeans(n_clusters=k, init='k-means++', n_init=5, 
                             bisecting_strategy='biggest_inertia', random_state=42).fit(X_blobs_scaled)
    bkm_lg = BisectingKMeans(n_clusters=k, init='k-means++', n_init=5, 
                             bisecting_strategy='largest_cluster', random_state=42).fit(X_blobs_scaled)
    
    inercia_lloyd.append(km.inertia_)
    inercia_bisect_inertia.append(bkm_in.inertia_)
    inercia_bisect_largest.append(bkm_lg.inertia_)

plt.figure(figsize=(11, 5.5))
plt.plot(rango_k, inercia_lloyd, 'o-', label='KMeans Estandar (Lloyd)', color='#1f77b4', lw=2)
plt.plot(rango_k, inercia_bisect_inertia, 's--', label='BisectingKMeans (biggest_inertia)', color='#d62728', lw=2)
plt.plot(rango_k, inercia_bisect_largest, '^:', label='BisectingKMeans (largest_cluster)', color='#2ca02c', lw=2)
plt.xlabel('Numero de Clusters (K)', fontsize=11, fontweight='bold')
plt.ylabel('Inercia Total (WCSS)', fontsize=11, fontweight='bold')
plt.title('Comparacion de Convergencia: KMeans vs. BisectingKMeans', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 2. Robustez Estadistica: K-Medoids (Algoritmo PAM)

El algoritmo **K-Medoids** (Kaufman & Rousseeuw, 1987 — *Partitioning Around Medoids, PAM*) impone una restriccion fundamental: **el centro de cada cluster debe ser obligatoriamente una observacion real del dataset ($m_k \in X$)**, denominada **Medoide** (*ejemplar o prototipo real*), y minimiza la suma de disimilaridades directas (como la distancia de Manhattan $L_1$ o cualquier matriz de distancias arbitraria $D_{ij}$):

$$J_{medoids}(C, m) = \sum_{k=1}^{K} \sum_{x_i \in C_k} d(x_i, m_k) \quad \text{sujeto a } \{m_1, \dots, m_K\} \subset X$$


In [ ]:
class KMedoidsPAM:
    """Implementacion vectorizada de K-Medoids (Voronoi Iteration / PAM) compatible con cualquier metrica de Scikit-Learn."""
    def __init__(self, n_clusters=3, metric='manhattan', max_iter=100, random_state=42):
        self.n_clusters = n_clusters
        self.metric = metric
        self.max_iter = max_iter
        self.random_state = random_state
        
    def fit(self, X_or_D, precomputed=False):
        rng = np.random.default_rng(self.random_state)
        D = X_or_D if precomputed else pairwise_distances(X_or_D, metric=self.metric)
        n = D.shape[0]
        
        medoid_indices = [rng.integers(0, n)]
        for _ in range(1, self.n_clusters):
            dist_min = D[:, medoid_indices].min(axis=1)
            probs = dist_min ** 2 / np.sum(dist_min ** 2)
            medoid_indices.append(rng.choice(n, p=probs))
        medoid_indices = np.array(medoid_indices)
        
        for _ in range(self.max_iter):
            labels = np.argmin(D[:, medoid_indices], axis=1)
            nuevos_medoides = np.copy(medoid_indices)
            for k in range(self.n_clusters):
                miembros_k = np.where(labels == k)[0]
                if len(miembros_k) > 0:
                    submatriz_D = D[np.ix_(miembros_k, miembros_k)]
                    costos_internos = submatriz_D.sum(axis=1)
                    nuevos_medoides[k] = miembros_k[np.argmin(costos_internos)]
                    
            if np.array_equal(medoid_indices, nuevos_medoides):
                break
            medoid_indices = nuevos_medoides
            
        self.medoid_indices_ = medoid_indices
        self.labels_ = np.argmin(D[:, medoid_indices], axis=1)
        self.inertia_ = np.sum(D[np.arange(n), medoid_indices[self.labels_]])
        if not precomputed:
            self.cluster_centers_ = np.asarray(X_or_D)[medoid_indices]
        return self
        
    def fit_predict(self, X_or_D, precomputed=False):
        return self.fit(X_or_D, precomputed=precomputed).labels_


In [ ]:
# Experimento de Robustez: Inyectamos 8 Outliers Extremos en 2D y comparamos Centroides vs Medoides
X_clean, y_clean = make_blobs(n_samples=300, centers=[[0, 0], [5, 5], [0, 6]], cluster_std=0.7, random_state=42)
outliers_extremos = np.array([[22, 22], [25, 20], [24, 24], [21, 19], [-18, -15], [-20, -18], [-19, -14], [23, 21]])
X_contaminado = np.vstack([X_clean, outliers_extremos])

km_cont = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_contaminado)
kmed_cont = KMedoidsPAM(n_clusters=3, metric='manhattan', random_state=42).fit(X_contaminado)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].scatter(X_contaminado[:, 0], X_contaminado[:, 1], c=km_cont.labels_, cmap='Set1', s=35, edgecolor='k', alpha=0.8)
axes[0].scatter(km_cont.cluster_centers_[:, 0], km_cont.cluster_centers_[:, 1], 
                c='black', marker='X', s=260, edgecolor='yellow', lw=2, label='Centroides (Media distorsionada)')
axes[0].set_xlim(-5, 10)
axes[0].set_ylim(-3, 10)
axes[0].set_title('K-Means: Los Outliers Arrastran el Centroide\ny Fusionan 2 Clusters Reales', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].scatter(X_contaminado[:, 0], X_contaminado[:, 1], c=kmed_cont.labels_, cmap='Set1', s=35, edgecolor='k', alpha=0.8)
axes[1].scatter(kmed_cont.cluster_centers_[:, 0], kmed_cont.cluster_centers_[:, 1], 
                c='cyan', marker='D', s=200, edgecolor='black', lw=2, label='Medoides Reales (Inmunes)')
axes[1].set_xlim(-5, 10)
axes[1].set_ylim(-3, 10)
axes[1].set_title('K-Medoids (PAM - L1): Los 3 Medoides Permanecen\nExactamente en el Centro de cada Nube', fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('Resistencia a Valores Atipicos Extremos (Zoom en la Zona Central)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"ARI sobre muestras legitimas - K-Means (contaminado):   {adjusted_rand_score(y_clean, km_cont.labels_[:300]):.4f}")
print(f"ARI sobre muestras legitimas - K-Medoids (robusto L1): {adjusted_rand_score(y_clean, kmed_cont.labels_[:300]):.4f}")


## 3. El Problema de los Datos Mixtos (Variables Numericas + Categoricas)

Existen **dos soluciones matematicamente rigurosas** para agrupar datos mixtos sin las distorsiones del *One-Hot Encoding*:
1. **Distancia de Gower (1971) + K-Medoids**: Combina Manhattan normalizada por el rango en variables continuas y distancia de Hamming en categoricas.
2. **Algoritmo K-Prototypes (Huang, 1998)**: Representa cada cluster mediante un vector de medias numericas $\mu_k$ y un vector de modas categoricas $M_k$, ponderados por el factor $\gamma$.


In [ ]:
# Creacion de un Dataset Bancario de Datos Mixtos (Numericos + Categoricos) con 3 Perfiles Reales
rng = np.random.default_rng(42)
n_por_grupo = 120

df_g0 = pd.DataFrame({
    'edad': rng.normal(22, 2.5, n_por_grupo).clip(18, 28),
    'ingreso_mensual_usd': rng.normal(850, 200, n_por_grupo).clip(400, 1500),
    'uso_tarjeta_pct': rng.normal(75, 10, n_por_grupo).clip(40, 100),
    'segmento_laboral': rng.choice(['Estudiante', 'Freelance', 'Empleado_Jr'], n_por_grupo, p=[0.75, 0.15, 0.10]),
    'canal_preferido': rng.choice(['App_Movil', 'Web', 'Sucursal'], n_por_grupo, p=[0.85, 0.12, 0.03]),
    'producto_estrella': rng.choice(['Debito_Cashback', 'Credito_Basica', 'Hipoteca'], n_por_grupo, p=[0.80, 0.18, 0.02])
})

df_g1 = pd.DataFrame({
    'edad': rng.normal(44, 5.0, n_por_grupo).clip(33, 58),
    'ingreso_mensual_usd': rng.normal(5200, 900, n_por_grupo).clip(3000, 8500),
    'uso_tarjeta_pct': rng.normal(45, 12, n_por_grupo).clip(15, 80),
    'segmento_laboral': rng.choice(['Ejecutivo_Sr', 'Empresario', 'Funcionario'], n_por_grupo, p=[0.65, 0.25, 0.10]),
    'canal_preferido': rng.choice(['Web', 'App_Movil', 'Sucursal'], n_por_grupo, p=[0.60, 0.30, 0.10]),
    'producto_estrella': rng.choice(['Hipoteca', 'Fondo_Inversion', 'Credito_Platinum'], n_por_grupo, p=[0.70, 0.20, 0.10])
})

df_g2 = pd.DataFrame({
    'edad': rng.normal(68, 4.5, n_por_grupo).clip(60, 82),
    'ingreso_mensual_usd': rng.normal(2400, 450, n_por_grupo).clip(1200, 4000),
    'uso_tarjeta_pct': rng.normal(18, 6, n_por_grupo).clip(2, 35),
    'segmento_laboral': rng.choice(['Pensionado', 'Rentista', 'Empresario'], n_por_grupo, p=[0.80, 0.15, 0.05]),
    'canal_preferido': rng.choice(['Sucursal', 'Web', 'App_Movil'], n_por_grupo, p=[0.82, 0.13, 0.05]),
    'producto_estrella': rng.choice(['Pagare_Plazo_Fijo', 'Fondo_Inversion', 'Hipoteca'], n_por_grupo, p=[0.85, 0.12, 0.03])
})

df_mixto = pd.concat([df_g0, df_g1, df_g2], ignore_index=True)
y_mixto_real = np.array([0]*n_por_grupo + [1]*n_por_grupo + [2]*n_por_grupo)

cols_num = ['edad', 'ingreso_mensual_usd', 'uso_tarjeta_pct']
cols_cat = ['segmento_laboral', 'canal_preferido', 'producto_estrella']
df_mixto.head()


In [ ]:
# 1. Calculo de la Matriz de Distancia de Gower Exacta
def calcular_matriz_gower(df, cols_numericas, cols_categoricas):
    n = len(df)
    p_total = len(cols_numericas) + len(cols_categoricas)
    D_acum = np.zeros((n, n), dtype=float)
    for col in cols_numericas:
        vals = df[col].values.reshape(-1, 1)
        rango = np.ptp(vals)
        if rango > 0:
            D_acum += pairwise_distances(vals, metric='manhattan') / rango
    for col in cols_categoricas:
        vals = df[col].values
        D_acum += (vals[:, None] != vals[None, :]).astype(float)
    return D_acum / p_total

# 2. Implementacion del Algoritmo K-Prototypes (Huang, 1998)
class KPrototypesCustom:
    def __init__(self, n_clusters=3, gamma=1.0, max_iter=50, random_state=42):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.max_iter = max_iter
        self.random_state = random_state
        
    def fit_predict(self, X_num, X_cat):
        rng = np.random.default_rng(self.random_state)
        n = X_num.shape[0]
        idx_init = rng.choice(n, size=self.n_clusters, replace=False)
        means_num = X_num[idx_init].copy()
        modes_cat = X_cat[idx_init].copy()
        labels = np.zeros(n, dtype=int)
        
        for _ in range(self.max_iter):
            dist_total = np.zeros((n, self.n_clusters))
            for k in range(self.n_clusters):
                d_num = np.sum((X_num - means_num[k]) ** 2, axis=1)
                d_cat = np.sum(X_cat != modes_cat[k], axis=1)
                dist_total[:, k] = d_num + self.gamma * d_cat
            nuevos_labels = np.argmin(dist_total, axis=1)
            if np.array_equal(labels, nuevos_labels):
                break
            labels = nuevos_labels
            for k in range(self.n_clusters):
                mask = (labels == k)
                if np.any(mask):
                    means_num[k] = X_num[mask].mean(axis=0)
                    for j in range(X_cat.shape[1]):
                        valores, conteos = np.unique(X_cat[mask, j], return_counts=True)
                        modes_cat[k, j] = valores[np.argmax(conteos)]
        self.means_num_ = means_num
        self.modes_cat_ = modes_cat
        self.labels_ = labels
        return labels

D_gower = calcular_matriz_gower(df_mixto, cols_num, cols_cat)
labels_gower_pam = KMedoidsPAM(n_clusters=3, random_state=42).fit_predict(D_gower, precomputed=True)

X_num_scaled = StandardScaler().fit_transform(df_mixto[cols_num])
X_cat_arr = df_mixto[cols_cat].values
kproto = KPrototypesCustom(n_clusters=3, gamma=0.8, random_state=42)
labels_kproto = kproto.fit_predict(X_num_scaled, X_cat_arr)

print(f"Exactitud de recuperacion (ARI) con Gower + K-Medoids: {adjusted_rand_score(y_mixto_real, labels_gower_pam):.4f}")
print(f"Exactitud de recuperacion (ARI) con K-Prototypes:      {adjusted_rand_score(y_mixto_real, labels_kproto):.4f}")
print("\nPrototipos Categoricos (Modas) descubiertos por K-Prototypes:")
print(pd.DataFrame(kproto.modes_cat_, columns=cols_cat, index=[f'Cluster {i}' for i in range(3)]))


## 4. Conclusiones

1. **`BisectingKMeans`**: Supera al K-Means clasico cuando se requiere una estructura jerarquica consistente o cuando el numero de clusters $K$ es grande.
2. **`K-Medoids` (PAM)**: Reemplaza la media aritmetica por el **medoide** (una observacion real del dataset que minimiza la suma de distancias $L_1$ o arbitrarias), otorgando robustez total frente a valores atipicos extremos.
3. **Datos Mixtos (Gower y K-Prototypes)**: Permiten agrupar tablas con variables continuas y categoricas nominales preservando la semantica de las categorias.
